# NetworkX で学ぶネットワーク分析 入門チュートリアル

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、Python のネットワーク分析ライブラリ
**NetworkX** の基礎を学ぶチュートリアルです。貿易ネットワーク、企業間取引、SNS、交通網など、
経済・社会の「つながり」を分析する道具を身につけます。

## 対象者
- Python と pandas の基本を理解している方
- ネットワーク分析（グラフ理論）を初めて学ぶ方
- 貿易・取引・人間関係などの「関係データ」を分析したい方

## このチュートリアルで学ぶこと
0. 環境準備（JupyterLite 用）
1. ネットワークとは：グラフの作成（無向・有向・重み付き）
2. グラフの描画
3. 次数と次数分布
4. 最短経路：都市間ネットワーク
5. 中心性：重要なノードを見つける
6. 連結性とネットワークの構造指標
7. コミュニティ検出
8. pandas との連携
9. ランダムグラフと経済への含意
10. まとめと総合演習

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。
- 各章の最後に **練習問題** があります。「解答欄」に自分でコードを書いてから、「解答例」を開いて確認しましょう。

---
## 0. 環境準備（JupyterLite 用）

NetworkX・pandas・matplotlib は JupyterLite に同梱されています。ノードのラベルに日本語を使うため
`japanize-matplotlib-jlite` を導入します。

In [ ]:
# JupyterLite 用のパッケージインストール
try:
    import piplite
    await piplite.install(["scipy", "networkx", "numpy", "pandas", "matplotlib", "japanize-matplotlib-jlite"])
except ImportError:
    pass

import scipy  # networkx の一部の関数（レイアウト・中心性・PageRank）が内部で使うため先に読み込む


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite  # 日本語表示用（plt の後に import する）

JP_FONT = "IPAexGothic"   # nx.draw のラベルに日本語を使うときに指定するフォント名
print(f"NetworkX バージョン: {nx.__version__}")
print(f"pandas バージョン: {pd.__version__}")

---
## 1. ネットワークとは：グラフの作成

ネットワーク（グラフ）は、**ノード**（点：人・企業・国・都市）と **エッジ**（線：取引・友人関係・貿易・道路）で表します。

| 種類 | クラス | 例 |
|---|---|---|
| 無向グラフ | `nx.Graph()` | 友人関係、共同研究（向きがない） |
| 有向グラフ | `nx.DiGraph()` | 輸出入、資金の流れ、Web リンク（向きがある） |
| 重み付きグラフ | エッジに `weight` 属性 | 貿易額、距離、取引回数 |

### 1.1 無向グラフを作る

In [ ]:
G = nx.Graph()
G.add_node("田中")
G.add_nodes_from(["鈴木", "佐藤", "高橋", "伊藤"])
G.add_edge("田中", "鈴木")
G.add_edges_from([("田中", "佐藤"), ("鈴木", "佐藤"), ("佐藤", "高橋"), ("高橋", "伊藤")])

print("ノード:", list(G.nodes))
print("エッジ:", list(G.edges))
print("ノード数:", G.number_of_nodes(), "エッジ数:", G.number_of_edges())

In [ ]:
# ノードの隣接（つながっている相手）を調べる
print("佐藤の友人:", list(G.neighbors("佐藤")))
print("田中と伊藤はつながっているか:", G.has_edge("田中", "伊藤"))
print("グラフの情報:", G)

### 1.2 有向グラフと重み付きエッジ

貿易は「A 国 → B 国へ輸出」という向きと「輸出額」をもつので、**重み付き有向グラフ** で表します。
エッジの属性はキーワード引数で自由に付けられます。

In [ ]:
trade = nx.DiGraph()
trade.add_edge("日本", "米国", weight=150)
trade.add_edge("日本", "中国", weight=140)
trade.add_edge("中国", "日本", weight=190)
trade.add_edge("中国", "米国", weight=500)
trade.add_edge("米国", "日本", weight=80)
trade.add_edge("米国", "中国", weight=150)
trade.add_edge("ドイツ", "米国", weight=130)
trade.add_edge("ドイツ", "中国", weight=100)
trade.add_edge("中国", "ドイツ", weight=120)

print("エッジと重み:")
for u, v, w in trade.edges(data="weight"):
    print(f"  {u} → {v}: {w} 億ドル")
print("日本の輸出先:", list(trade.successors("日本")))
print("日本への輸出国:", list(trade.predecessors("日本")))

### 1.3 ノードやエッジに属性を付ける

In [ ]:
trade.nodes["日本"]["gdp"] = 4.2
trade.nodes["米国"]["gdp"] = 25.5
trade.nodes["中国"]["gdp"] = 17.9
trade.nodes["ドイツ"]["gdp"] = 4.1

print(trade.nodes(data=True))
print("日本→米国の重み:", trade["日本"]["米国"]["weight"])
print("日本の総輸出額:", trade.out_degree("日本", weight="weight"))
print("日本の総輸入額:", trade.in_degree("日本", weight="weight"))

### 練習問題 1

1. 5 つの都市（東京、名古屋、大阪、福岡、札幌）をノードとし、東京–名古屋、名古屋–大阪、東京–大阪、東京–札幌、大阪–福岡を結ぶ無向グラフ `cities` を作り、ノード数・エッジ数を表示してください。
2. `cities` で「東京」の隣接都市を表示してください。
3. 3 社（A社, B社, C社）の間の取引を有向グラフで作ってください：A→B に 100、B→C に 60、C→A に 30（`weight`）。各社の「売上（out_degree の重み付き）」を表示してください。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
# 1
cities = nx.Graph()
cities.add_edges_from([("東京", "名古屋"), ("名古屋", "大阪"), ("東京", "大阪"), ("東京", "札幌"), ("大阪", "福岡")])
print(cities.number_of_nodes(), cities.number_of_edges())

# 2
print(list(cities.neighbors("東京")))

# 3
firms = nx.DiGraph()
firms.add_weighted_edges_from([("A社", "B社", 100), ("B社", "C社", 60), ("C社", "A社", 30)])
for f in firms.nodes:
    print(f, firms.out_degree(f, weight="weight"))
```

</details>

---
## 2. グラフの描画

`nx.draw()` でグラフを描けます。ノードの配置（レイアウト）は `pos` で指定します。
`nx.spring_layout(G, seed=...)` は「つながりの強いノードが近くに来る」配置で、`seed` を固定すると毎回同じ図になります。
日本語ラベルには `font_family=JP_FONT` を指定します。

In [ ]:
pos = nx.spring_layout(G, seed=1)
plt.figure(figsize=(6, 4.5))
nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=1500, font_family=JP_FONT, font_size=12)
plt.title("友人関係ネットワーク")
plt.show()

### 2.1 レイアウトを変える

| レイアウト | 特徴 |
|---|---|
| `spring_layout` | ばねモデル。一般的なネットワーク向け |
| `circular_layout` | 円周上に配置。小さなグラフの構造確認に |
| `shell_layout` | 同心円 |
| `kamada_kawai_layout` | 距離を保つように配置。見やすいことが多い |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
layouts = {"spring": nx.spring_layout(G, seed=1), "circular": nx.circular_layout(G), "kamada_kawai": nx.kamada_kawai_layout(G)}
for ax, (name, layout) in zip(axes, layouts.items()):
    nx.draw(G, layout, ax=ax, with_labels=True, node_color="lightgreen", node_size=1000, font_family=JP_FONT)
    ax.set_title(name)
plt.show()

### 2.2 有向グラフと重みの表示

有向グラフは矢印で描かれます。`nx.draw_networkx_edge_labels()` でエッジの重みを表示し、
`width` に重みを渡すと線の太さで取引額を表現できます。

In [ ]:
pos_t = nx.circular_layout(trade)
weights = [trade[u][v]["weight"] / 100 for u, v in trade.edges]

plt.figure(figsize=(7, 6))
nx.draw(trade, pos_t, with_labels=True, node_color="gold", node_size=2200, font_family=JP_FONT,
        width=weights, arrowsize=20, connectionstyle="arc3,rad=0.15")
nx.draw_networkx_edge_labels(trade, pos_t, edge_labels={(u, v): w for u, v, w in trade.edges(data="weight")},
                             font_family=JP_FONT, label_pos=0.3)
plt.title("貿易ネットワーク（線の太さ = 輸出額）")
plt.show()

### 練習問題 2

1. 練習問題 1 の `cities` を `kamada_kawai_layout` で描いてください（日本語ラベル付き）。
2. `cities` のエッジに距離（km）を重みとして追加し（東京–名古屋 350、名古屋–大阪 190、東京–大阪 500、東京–札幌 830、大阪–福岡 610）、エッジラベルとして表示してください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
# 1
pos_c = nx.kamada_kawai_layout(cities)
nx.draw(cities, pos_c, with_labels=True, node_color="lightblue", node_size=1500, font_family=JP_FONT)
plt.show()

# 2
dist = {("東京", "名古屋"): 350, ("名古屋", "大阪"): 190, ("東京", "大阪"): 500, ("東京", "札幌"): 830, ("大阪", "福岡"): 610}
nx.set_edge_attributes(cities, dist, "weight")
nx.draw(cities, pos_c, with_labels=True, node_color="lightblue", node_size=1500, font_family=JP_FONT)
nx.draw_networkx_edge_labels(cities, pos_c, edge_labels=dist)
plt.show()
```

</details>

---
## 3. 次数と次数分布

**次数（degree）** は、そのノードにつながるエッジの数です。友人の数、取引先の数、貿易相手国の数に相当します。
有向グラフでは **入次数（in-degree）** と **出次数（out-degree）** に分かれます。

In [ ]:
print("次数:", dict(G.degree))
print("重み付き次数（貿易総額）:", dict(trade.degree(weight="weight")))
print("入次数（輸入相手国の数）:", dict(trade.in_degree))
print("出次数（輸出相手国の数）:", dict(trade.out_degree))

### 3.1 次数分布

大きなネットワークでは、次数の **分布** を見ます。少数のノードに多くのエッジが集中する（ハブがある）のが
社会・経済ネットワークの典型的な特徴です。ここでは合成データで企業間取引ネットワークを作って調べます。

In [ ]:
# 100 社の取引ネットワークを生成（優先的選択：取引先の多い企業ほど新しい取引先を得やすい）
firm_net = nx.barabasi_albert_graph(n=100, m=2, seed=7)
degrees = [d for _, d in firm_net.degree]

print("平均次数:", np.mean(degrees))
print("最大次数:", max(degrees), "→ ハブ企業のノード番号:", max(firm_net.degree, key=lambda x: x[1])[0])
print("密度（実際のエッジ数 / 可能なエッジ数）:", round(nx.density(firm_net), 4))

plt.figure(figsize=(7, 4))
plt.hist(degrees, bins=range(1, max(degrees) + 2), edgecolor="black")
plt.xlabel("次数（取引先の数）")
plt.ylabel("企業数")
plt.title("企業間取引ネットワークの次数分布")
plt.grid(True, axis="y")
plt.show()

### 練習問題 3

1. `trade` の各国について、輸出額の合計（`out_degree(weight="weight")`）を大きい順に表示してください。
2. `firm_net` で次数が 10 以上の企業（ノード番号）をリストアップしてください。
3. `firm_net` の次数の中央値と、次数 1〜3 の企業の割合を求めてください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
print(sorted(trade.out_degree(weight="weight"), key=lambda x: x[1], reverse=True))

# 2
print([n for n, d in firm_net.degree if d >= 10])

# 3
deg = np.array(degrees)
print(np.median(deg), np.mean(deg <= 3))
```

</details>

---
## 4. 最短経路：都市間ネットワーク

都市を結ぶ道路網で、ある都市から別の都市への **最短経路** を求めます。
`nx.shortest_path(G, 出発, 到着, weight="weight")` は重み（距離）の合計が最小の経路を返します
（内部ではダイクストラ法が使われます）。重みを指定しなければ「通過するエッジ数」が最小の経路になります。

In [ ]:
roads = nx.Graph()
roads.add_weighted_edges_from([
    ("札幌", "仙台", 800), ("仙台", "東京", 350), ("東京", "名古屋", 350), ("東京", "金沢", 450),
    ("名古屋", "大阪", 190), ("金沢", "大阪", 300), ("大阪", "広島", 330), ("大阪", "高松", 200),
    ("広島", "福岡", 280), ("高松", "福岡", 400), ("名古屋", "金沢", 250),
])

path = nx.shortest_path(roads, "東京", "福岡", weight="weight")
length = nx.shortest_path_length(roads, "東京", "福岡", weight="weight")
print("東京 → 福岡 の最短経路:", " → ".join(path), f"（{length} km）")

path_hops = nx.shortest_path(roads, "東京", "福岡")   # 重みなし = 経由都市数が最小
print("経由数が最小の経路:", " → ".join(path_hops))

In [ ]:
# 最短経路を強調して描画
pos_r = nx.kamada_kawai_layout(roads, weight="weight")
path_edges = list(zip(path[:-1], path[1:]))

plt.figure(figsize=(8, 6))
nx.draw(roads, pos_r, with_labels=True, node_color="lightgray", node_size=1400, font_family=JP_FONT)
nx.draw_networkx_edges(roads, pos_r, edgelist=path_edges, edge_color="red", width=3)
nx.draw_networkx_edge_labels(roads, pos_r, edge_labels=nx.get_edge_attributes(roads, "weight"), font_size=8)
plt.title("東京 → 福岡 の最短経路（赤）")
plt.show()

### 4.1 すべての都市からの距離

`nx.single_source_dijkstra_path_length()` で、ある都市から全都市への最短距離が一度に求まります。
物流拠点の立地（どこに倉庫を置けば全体の輸送距離が短いか）の検討に使えます。

In [ ]:
for hub in ("東京", "大阪", "名古屋"):
    dist = nx.single_source_dijkstra_path_length(roads, hub, weight="weight")
    print(f"{hub} を拠点にした場合の総距離: {sum(dist.values())} km, 最も遠い都市: {max(dist, key=dist.get)} ({max(dist.values())} km)")

### 練習問題 4

1. `roads` で札幌から広島までの最短経路と距離を求めてください。
2. 「東京–金沢」の道路が通行止めになった（エッジを削除した）ときの、東京から金沢への最短経路を求めてください（`roads.copy()` してから `remove_edge`）。
3. 各都市を拠点にしたときの「全都市への距離の合計」を計算し、最も小さい都市（最適な物流拠点）を求めてください。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
# 1
print(nx.shortest_path(roads, "札幌", "広島", weight="weight"), nx.shortest_path_length(roads, "札幌", "広島", weight="weight"))

# 2
closed = roads.copy()
closed.remove_edge("東京", "金沢")
print(nx.shortest_path(closed, "東京", "金沢", weight="weight"))

# 3
totals = {c: sum(nx.single_source_dijkstra_path_length(roads, c, weight="weight").values()) for c in roads.nodes}
print(min(totals, key=totals.get), totals)
```

</details>

---
## 5. 中心性：重要なノードを見つける

「ネットワークの中で重要なノードはどれか」を測る指標を **中心性（centrality）** といいます。
何をもって「重要」とするかで、複数の指標があります。

| 中心性 | 意味 | 経済での解釈 |
|---|---|---|
| 次数中心性 `degree_centrality` | つながりの数 | 取引先の多さ |
| 媒介中心性 `betweenness_centrality` | 他のノード間の最短経路上にどれだけ現れるか | 仲介者・ボトルネック（物流のハブ、金融の中枢） |
| 近接中心性 `closeness_centrality` | 全ノードへの距離の近さ | 情報や商品が速く届く位置 |
| 固有ベクトル中心性 `eigenvector_centrality` | 重要なノードとつながっているほど高い | 有力企業と取引している企業 |
| PageRank `pagerank` | 有向グラフ版の固有ベクトル中心性 | 資金・貿易の流れの中の影響力 |

### 5.1 企業間取引ネットワークでの中心性

30 社の取引ネットワークを作り、各中心性を計算して DataFrame にまとめます。

In [ ]:
rng = np.random.default_rng(3)
supply_net = nx.DiGraph()
firms = [f"企業{i:02d}" for i in range(1, 31)]
supply_net.add_nodes_from(firms)
# 優先的選択で「取引先の多い企業」が生まれるようにエッジを追加
for i, buyer in enumerate(firms[1:], start=1):
    n_suppliers = rng.integers(1, 4)
    weights = np.array([supply_net.degree(f) + 1 for f in firms[:i]], dtype=float)
    suppliers = rng.choice(firms[:i], size=min(n_suppliers, i), replace=False, p=weights / weights.sum())
    for s in suppliers:
        supply_net.add_edge(s, buyer, weight=int(rng.integers(10, 100)))   # 仕入先 → 買い手（金額）

print(supply_net)

In [ ]:
centrality = pd.DataFrame({
    "次数": dict(nx.degree_centrality(supply_net)),
    "媒介": dict(nx.betweenness_centrality(supply_net)),
    "近接": dict(nx.closeness_centrality(supply_net)),
    "固有ベクトル": dict(nx.eigenvector_centrality(supply_net.to_undirected(), max_iter=1000)),
    "PageRank": dict(nx.pagerank(supply_net, weight="weight")),
}).round(3)

print(centrality.sort_values("PageRank", ascending=False).head(8))

指標によって上位企業が変わることに注目してください。
たとえば媒介中心性が高い企業は、取引先の数が少なくても「その企業が倒産すると多くの取引経路が途切れる」重要企業です。

In [ ]:
# 各指標の上位 3 社
for col in centrality.columns:
    top = centrality[col].sort_values(ascending=False).head(3)
    print(f"{col:8s}: " + ", ".join(f"{name}({v})" for name, v in top.items()))

In [ ]:
# PageRank の大きさでノードサイズを変えて描画
pos_s = nx.spring_layout(supply_net, seed=5, k=0.6)
sizes = [3000 * v for v in centrality["PageRank"]]

plt.figure(figsize=(10, 8))
nx.draw(supply_net, pos_s, with_labels=True, node_size=sizes, node_color=centrality["媒介"], cmap="YlOrRd",
        font_family=JP_FONT, font_size=8, arrowsize=10, edge_color="gray")
plt.title("企業間取引ネットワーク（大きさ = PageRank、色 = 媒介中心性）")
plt.show()

### 練習問題 5

1. `trade`（貿易ネットワーク）の PageRank を `weight="weight"` 付きで計算し、大きい順に表示してください。
2. `roads`（道路網）の媒介中心性を計算し、最も高い都市（物流のボトルネック）を答えてください。
3. `supply_net` で「媒介中心性の順位」と「次数中心性の順位」が最も食い違う企業を見つけてください（ヒント: `centrality.rank(ascending=False)` で順位を求め、差の絶対値を計算）。

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```python
# 1
print(sorted(nx.pagerank(trade, weight="weight").items(), key=lambda x: x[1], reverse=True))

# 2
bc = nx.betweenness_centrality(roads, weight="weight")
print(max(bc, key=bc.get), round(bc[max(bc, key=bc.get)], 3))

# 3
ranks = centrality.rank(ascending=False)
gap = (ranks["媒介"] - ranks["次数"]).abs()
print(gap.idxmax(), gap.max())
```

</details>

---
## 6. 連結性とネットワークの構造指標

### 6.1 連結成分

ネットワークが「ひとつながり」になっているかを **連結性** といいます。
バラバラの部分（連結成分）に分かれている場合、どの部分にも属さないノードには商品や情報が届きません。

In [ ]:
fragmented = nx.Graph()
fragmented.add_edges_from([(1, 2), (2, 3), (3, 1), (4, 5), (6, 7), (7, 8)])
fragmented.add_node(9)

print("連結か:", nx.is_connected(fragmented))
print("連結成分の数:", nx.number_connected_components(fragmented))
print("各成分:", [sorted(c) for c in nx.connected_components(fragmented)])
largest = max(nx.connected_components(fragmented), key=len)
print("最大の成分:", sorted(largest))

### 6.2 構造指標：クラスター係数・平均最短経路長・直径

| 指標 | 意味 |
|---|---|
| `average_clustering` | 「友人の友人が友人である」割合（三角形の多さ） |
| `average_shortest_path_length` | 任意の 2 ノード間の平均距離（小さいほど「世間は狭い」） |
| `diameter` | 最も遠い 2 ノード間の距離 |

In [ ]:
for name, graph in [("友人関係", G), ("道路網", roads), ("企業取引（無向）", firm_net)]:
    print(f"{name}: クラスター係数 = {nx.average_clustering(graph):.3f}, "
          f"平均最短経路長 = {nx.average_shortest_path_length(graph):.2f}, 直径 = {nx.diameter(graph)}")

### 6.3 ノードを取り除くとどうなるか（頑健性）

ハブ企業が倒産したり、主要な港が閉鎖されたりすると、ネットワークがどれだけ分断されるかを調べます。

In [ ]:
hub = max(firm_net.degree, key=lambda x: x[1])[0]
without_hub = firm_net.copy()
without_hub.remove_node(hub)
random_node = 50
without_random = firm_net.copy()
without_random.remove_node(random_node)

print(f"元のネットワーク: 連結成分 {nx.number_connected_components(firm_net)}, 最大成分の大きさ {len(max(nx.connected_components(firm_net), key=len))}")
print(f"ハブ企業 {hub} を除去: 連結成分 {nx.number_connected_components(without_hub)}, 最大成分の大きさ {len(max(nx.connected_components(without_hub), key=len))}")
print(f"企業 {random_node} を除去: 連結成分 {nx.number_connected_components(without_random)}, 最大成分の大きさ {len(max(nx.connected_components(without_random), key=len))}")

### 練習問題 6

1. `roads` から「東京」を取り除いたとき、ネットワークは連結のままか調べ、連結成分を表示してください。
2. `roads` で最も遠い 2 都市の組み合わせ（直径を実現するペア）を `nx.periphery` で求めてください。
3. `firm_net` で次数の高い上位 5 社を順に取り除いていき、そのたびに最大連結成分の大きさを表示してください。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```python
# 1
r2 = roads.copy(); r2.remove_node("東京")
print(nx.is_connected(r2), [sorted(c) for c in nx.connected_components(r2)])

# 2
print(nx.periphery(roads), nx.diameter(roads))

# 3
g = firm_net.copy()
for n, _ in sorted(firm_net.degree, key=lambda x: x[1], reverse=True)[:5]:
    g.remove_node(n)
    print(n, len(max(nx.connected_components(g), key=len)))
```

</details>

---
## 7. コミュニティ検出

**コミュニティ** は、内部のつながりが密で、外部とのつながりが疎なノードのまとまりです。
産業クラスター、貿易ブロック、SNS の派閥などに対応します。
`nx.community.greedy_modularity_communities()` は、モジュラリティ（コミュニティ分割の良さ）を最大化する分割を求めます。

例として、有名な「空手クラブ」のネットワーク（34 人の友人関係。後に 2 派に分裂した）を使います。

In [ ]:
karate = nx.karate_club_graph()
communities = nx.community.greedy_modularity_communities(karate)
print("コミュニティ数:", len(communities))
for i, c in enumerate(communities):
    print(f"  コミュニティ {i}: {sorted(c)}")
print("モジュラリティ:", round(nx.community.modularity(karate, communities), 3))

In [ ]:
# コミュニティごとに色分けして描画
color_map = {}
for i, c in enumerate(communities):
    for node in c:
        color_map[node] = i
node_colors = [color_map[n] for n in karate.nodes]

pos_k = nx.spring_layout(karate, seed=4)
plt.figure(figsize=(8, 6))
nx.draw(karate, pos_k, with_labels=True, node_color=node_colors, cmap="Set2", node_size=500, font_size=8)
plt.title("空手クラブネットワークのコミュニティ")
plt.show()

### 7.1 実際の分裂と比較する

ノード属性 `club` に、実際にどちらの派閥に入ったかが記録されています。検出したコミュニティと照らし合わせてみましょう。

In [ ]:
actual = pd.Series(nx.get_node_attributes(karate, "club"))
detected = pd.Series(color_map)
print(pd.crosstab(actual, detected, rownames=["実際の所属"], colnames=["検出コミュニティ"]))

### 練習問題 7

1. `supply_net` を無向グラフに変換（`to_undirected()`）してコミュニティ検出を行い、コミュニティ数と各コミュニティの企業を表示してください。
2. `firm_net` のコミュニティを検出し、最も大きいコミュニティの企業数とモジュラリティを表示してください。

In [ ]:
# 練習問題 7 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 7 の解答例を見る</strong></summary>

```python
# 1
comms = nx.community.greedy_modularity_communities(supply_net.to_undirected())
print(len(comms))
for c in comms:
    print(sorted(c))

# 2
comms2 = nx.community.greedy_modularity_communities(firm_net)
print(len(max(comms2, key=len)), round(nx.community.modularity(firm_net, comms2), 3))
```

</details>

---
## 8. pandas との連携

実際のデータは「取引元・取引先・金額」のような **エッジリスト** の表（CSV）で手に入ることが多いです。
`nx.from_pandas_edgelist()` で DataFrame からグラフを作り、`nx.to_pandas_edgelist()` で戻せます。

In [ ]:
edges_df = pd.DataFrame({
    "輸出国": ["日本", "日本", "中国", "中国", "米国", "米国", "ドイツ", "ドイツ", "中国", "韓国", "韓国"],
    "輸入国": ["米国", "中国", "日本", "米国", "日本", "中国", "米国", "中国", "ドイツ", "日本", "中国"],
    "金額": [150, 140, 190, 500, 80, 150, 130, 100, 120, 60, 130],
})
print(edges_df.head())

trade2 = nx.from_pandas_edgelist(edges_df, source="輸出国", target="輸入国", edge_attr="金額", create_using=nx.DiGraph)
print(trade2)

In [ ]:
# グラフの計算結果を DataFrame に戻して集計
summary = pd.DataFrame({
    "輸出額": dict(trade2.out_degree(weight="金額")),
    "輸入額": dict(trade2.in_degree(weight="金額")),
    "PageRank": dict(nx.pagerank(trade2, weight="金額")),
})
summary["貿易収支"] = summary["輸出額"] - summary["輸入額"]
print(summary.round(3).sort_values("輸出額", ascending=False))

# エッジリストに戻す
print(nx.to_pandas_edgelist(trade2).head())

### 8.1 ノード属性を DataFrame から付ける

ノードの属性（GDP、業種、所在地など）も DataFrame から `set_node_attributes()` で付けられます。

In [ ]:
nodes_df = pd.DataFrame({"国": ["日本", "米国", "中国", "ドイツ", "韓国"], "GDP": [4.2, 25.5, 17.9, 4.1, 1.7]}).set_index("国")
nx.set_node_attributes(trade2, nodes_df["GDP"].to_dict(), "GDP")

# GDP に対する輸出額の比率（貿易依存度）を計算
for country in trade2.nodes:
    ratio = trade2.out_degree(country, weight="金額") / (trade2.nodes[country]["GDP"] * 1000)
    print(f"{country}: 輸出/GDP = {ratio:.3f}")

### 練習問題 8

1. `edges_df` から「金額が 100 以上」のエッジだけを使ってグラフを作り、エッジ数を表示してください。
2. `trade2` の各国について「貿易相手国数」（入次数 + 出次数の相手の集合の大きさ、`set(successors) | set(predecessors)`）を DataFrame にまとめてください。
3. `trade2` を `nx.to_pandas_edgelist()` で DataFrame に戻し、金額の大きい順に並べ替えて上位 3 件を表示してください。

In [ ]:
# 練習問題 8 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 8 の解答例を見る</strong></summary>

```python
# 1
big = nx.from_pandas_edgelist(edges_df[edges_df["金額"] >= 100], source="輸出国", target="輸入国", edge_attr="金額", create_using=nx.DiGraph)
print(big.number_of_edges())

# 2
partners = {c: len(set(trade2.successors(c)) | set(trade2.predecessors(c))) for c in trade2.nodes}
print(pd.DataFrame({"相手国数": partners}))

# 3
print(nx.to_pandas_edgelist(trade2).sort_values("金額", ascending=False).head(3))
```

</details>

---
## 9. ランダムグラフと経済への含意

現実のネットワークの特徴を理解するために、単純なルールで作った **ランダムグラフ** と比較します。

| モデル | 関数 | 特徴 |
|---|---|---|
| エルデシュ＝レニイ | `erdos_renyi_graph(n, p)` | 各ペアが確率 $p$ でつながる。次数はポアソン分布（ハブが生まれない） |
| バラバシ＝アルバート | `barabasi_albert_graph(n, m)` | 優先的選択。次数はべき乗則（少数のハブに集中） |
| ワッツ＝ストロガッツ | `watts_strogatz_graph(n, k, p)` | スモールワールド（クラスターが多いのに距離が短い） |

経済ネットワーク（企業間取引、銀行間市場、貿易）は、バラバシ＝アルバート型に近い **スケールフリー** な構造をもつことが多く、
「少数のハブに依存する」＝「ハブの破綻が全体に波及する」というシステミック・リスクの議論につながります。

In [ ]:
n = 300
er = nx.erdos_renyi_graph(n, p=4 / n, seed=1)
ba = nx.barabasi_albert_graph(n, m=2, seed=1)
ws = nx.watts_strogatz_graph(n, k=4, p=0.1, seed=1)

for name, graph in [("エルデシュ＝レニイ", er), ("バラバシ＝アルバート", ba), ("ワッツ＝ストロガッツ", ws)]:
    degs = [d for _, d in graph.degree]
    giant = graph.subgraph(max(nx.connected_components(graph), key=len))
    print(f"{name}: 平均次数 {np.mean(degs):.2f}, 最大次数 {max(degs)}, クラスター係数 {nx.average_clustering(graph):.3f}, "
          f"平均最短経路長 {nx.average_shortest_path_length(giant):.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, graph) in zip(axes, [("エルデシュ＝レニイ", er), ("バラバシ＝アルバート", ba), ("ワッツ＝ストロガッツ", ws)]):
    degs = [d for _, d in graph.degree]
    ax.hist(degs, bins=range(0, max(degs) + 2), edgecolor="black")
    ax.set_title(name)
    ax.set_xlabel("次数")
    ax.set_ylabel("ノード数")
plt.suptitle("ランダムグラフの次数分布の違い")
plt.tight_layout()
plt.show()

### 9.1 べき乗則の確認（両対数グラフ）

スケールフリー・ネットワークでは、次数 $k$ をもつノードの割合が $k^{-\gamma}$ に従うため、両対数グラフで直線に近くなります。

In [ ]:
degs_ba = np.array([d for _, d in ba.degree])
values, counts = np.unique(degs_ba, return_counts=True)

plt.figure(figsize=(6, 4.5))
plt.loglog(values, counts / counts.sum(), "o", label="バラバシ＝アルバート")
degs_er = np.array([d for _, d in er.degree])
v2, c2 = np.unique(degs_er, return_counts=True)
plt.loglog(v2, c2 / c2.sum(), "s", label="エルデシュ＝レニイ")
plt.xlabel("次数 k（対数）")
plt.ylabel("割合 P(k)（対数）")
plt.title("次数分布の両対数プロット")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

### 練習問題 9

1. `barabasi_albert_graph(500, m=3, seed=2)` を作り、次数上位 5 ノードが全エッジの何 % を占めるかを計算してください（ノードの次数の合計 = エッジ数の 2 倍）。
2. `erdos_renyi_graph(500, p=0.012, seed=2)` について同じ計算をし、比べてください。
3. ワッツ＝ストロガッツ・モデルで `p` を 0, 0.1, 1 と変えたとき、クラスター係数と平均最短経路長がどう変わるか表にしてください（`n=200, k=4`）。

In [ ]:
# 練習問題 9 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 9 の解答例を見る</strong></summary>

```python
# 1
ba2 = nx.barabasi_albert_graph(500, m=3, seed=2)
top5 = sorted(d for _, d in ba2.degree)[-5:]
print(sum(top5) / (2 * ba2.number_of_edges()))

# 2
er2 = nx.erdos_renyi_graph(500, p=0.012, seed=2)
top5e = sorted(d for _, d in er2.degree)[-5:]
print(sum(top5e) / (2 * er2.number_of_edges()))

# 3
for p in (0, 0.1, 1):
    g = nx.watts_strogatz_graph(200, 4, p, seed=1)
    giant = g.subgraph(max(nx.connected_components(g), key=len))
    print(p, round(nx.average_clustering(g), 3), round(nx.average_shortest_path_length(giant), 2))
```

</details>

---
## 10. まとめ

| トピック | 主な関数 |
|---|---|
| グラフの作成 | `nx.Graph()`, `nx.DiGraph()`, `add_edge()`, `add_weighted_edges_from()`, `nodes`, `edges`, `neighbors()` |
| 描画 | `nx.draw()`, `spring_layout()`, `kamada_kawai_layout()`, `draw_networkx_edge_labels()` |
| 次数 | `degree`, `in_degree`, `out_degree`, `degree(weight=)`, `density()` |
| 最短経路 | `shortest_path()`, `shortest_path_length()`, `single_source_dijkstra_path_length()` |
| 中心性 | `degree_centrality()`, `betweenness_centrality()`, `closeness_centrality()`, `eigenvector_centrality()`, `pagerank()` |
| 連結性・構造 | `is_connected()`, `connected_components()`, `average_clustering()`, `average_shortest_path_length()`, `diameter()` |
| コミュニティ | `nx.community.greedy_modularity_communities()`, `nx.community.modularity()` |
| pandas 連携 | `from_pandas_edgelist()`, `to_pandas_edgelist()`, `set_node_attributes()` |
| ランダムグラフ | `erdos_renyi_graph()`, `barabasi_albert_graph()`, `watts_strogatz_graph()` |

## 次のステップ

- `python/pyvis/pyvis_beginner_tutorial.ipynb` — ネットワークをマウスで動かせる対話的な図にする
- `python/pandas/pandas_intermediate_tutorial.ipynb` — 取引データの集計・結合でエッジリストを作る
- `python/folium/geopandas_folium_beginner_tutorial.ipynb` — 都市間ネットワークを地図の上に描く

---
## 総合演習：企業間取引ネットワークの分析

次のセルで生成する 40 社の取引ネットワーク `biz`（有向・重み付き：仕入先 → 買い手、重みは取引額）について分析してください。

1. ノード数・エッジ数・密度を表示し、各企業の「販売額（out_degree の重み付き）」と「仕入額（in_degree の重み付き）」を DataFrame にまとめて、販売額の上位 5 社を表示してください。
2. 次数中心性・媒介中心性・PageRank（重み付き）を計算し、それぞれの上位 3 社を表示してください。
3. 無向グラフに変換してコミュニティ検出を行い、コミュニティ数とモジュラリティを表示してください。
4. 販売額が最大の企業が倒産した（ノードを除去した）とき、無向グラフの連結成分数と最大成分の大きさがどう変わるか調べてください。
5. PageRank でノードの大きさ、コミュニティで色を変えてネットワークを描画してください。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください（まずこのセルを実行してデータを作る）
rng = np.random.default_rng(2026)
biz = nx.DiGraph()
firm_names = [f"F{i:02d}" for i in range(1, 41)]
biz.add_nodes_from(firm_names)
for i, buyer in enumerate(firm_names[1:], start=1):
    weights = np.array([biz.degree(f) + 1 for f in firm_names[:i]], dtype=float)
    for supplier in rng.choice(firm_names[:i], size=min(int(rng.integers(1, 4)), i), replace=False, p=weights / weights.sum()):
        biz.add_edge(supplier, buyer, weight=int(rng.integers(5, 200)))
print(biz)

### 総合演習の解答例

自分で書いてから、次のセルを実行して結果を比べてみてください。

In [ ]:
# 1. 基本統計
print("ノード数:", biz.number_of_nodes(), "エッジ数:", biz.number_of_edges(), "密度:", round(nx.density(biz), 3))
sales = pd.DataFrame({"販売額": dict(biz.out_degree(weight="weight")), "仕入額": dict(biz.in_degree(weight="weight"))})
print(sales.sort_values("販売額", ascending=False).head())

# 2. 中心性
cent = pd.DataFrame({
    "次数": nx.degree_centrality(biz),
    "媒介": nx.betweenness_centrality(biz),
    "PageRank": nx.pagerank(biz, weight="weight"),
}).round(3)
for col in cent.columns:
    print(col, cent[col].sort_values(ascending=False).head(3).to_dict())

# 3. コミュニティ
biz_u = biz.to_undirected()
comms = nx.community.greedy_modularity_communities(biz_u)
print("コミュニティ数:", len(comms), "モジュラリティ:", round(nx.community.modularity(biz_u, comms), 3))

# 4. 最大企業の倒産
top_firm = sales["販売額"].idxmax()
damaged = biz_u.copy()
damaged.remove_node(top_firm)
print(f"{top_firm} 除去前: 成分数 {nx.number_connected_components(biz_u)}, 最大成分 {len(max(nx.connected_components(biz_u), key=len))}")
print(f"{top_firm} 除去後: 成分数 {nx.number_connected_components(damaged)}, 最大成分 {len(max(nx.connected_components(damaged), key=len))}")

# 5. 描画
color_of = {node: i for i, c in enumerate(comms) for node in c}
pos_b = nx.spring_layout(biz, seed=9, k=0.5)
plt.figure(figsize=(10, 8))
nx.draw(biz, pos_b, with_labels=True, node_size=[4000 * v for v in cent["PageRank"]],
        node_color=[color_of[n] for n in biz.nodes], cmap="Set3", font_size=8, arrowsize=8, edge_color="gray")
plt.title("企業間取引ネットワーク（大きさ = PageRank、色 = コミュニティ）")
plt.show()

お疲れさまでした！ ネットワーク分析は「関係のデータ」を扱う強力な道具です。
貿易統計や企業の取引データ、SNS のフォロー関係など、身近なデータをエッジリストにして分析してみましょう。